# Lab 14 - Evaluation Metrics: How Do You Know Your Agent Is Working?
**Section covered: 13 (Evaluation Metrics)**

---

## What this lab builds

Section 13 introduced the three-level evaluation framework:
- **System-level** - latency, tokens, cost, error rate (operational health)
- **Task-level** - success rate, completion rate, human intervention rate (usefulness)
- **Tool-level** - selection accuracy, parameter accuracy, success rate (diagnostic)

This lab builds an **evaluation harness** that:
1. Instruments the agent loop to collect all three levels of metrics
2. Runs an evaluation suite of 8 test tasks with golden-standard expectations
3. Generates a full evaluation report with diagnostic interpretations
4. Shows how to push metrics to Amazon CloudWatch in production

---

## Why evaluation matters for agents

For a static LLM: did the output match the expected answer? Straightforward.

For an agent: you need to measure across the entire execution chain.
The agent might give the right final answer while calling the wrong tool,
passing wrong parameters, or taking 10x more turns than necessary.
Each of those failures means something different and needs a different fix.

---

## The human intervention rate

This is the single most important metric. It answers:
"What fraction of tasks did a human have to finish because the agent couldn't?"

- Below 20%: agent is genuinely autonomous
- 20-40%: agent is a supervised assistant - review failure patterns
- Above 40%: you have not automated the task, you have added a layer of complexity

## Step 1 - Setup

In [ ]:
import anthropic
import json
import time
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
print("Client ready")
print("This lab builds a full evaluation harness covering all three metric levels:")
print("  System-level  - latency, cost, error rate")
print("  Task-level    - success rate, completion rate, human intervention rate")
print("  Tool-level    - selection accuracy, parameter accuracy, success rate")

## Step 2 - The Metrics Collector

Collects metrics at all three evaluation levels during every agent run.

In [ ]:
# ===================================================================
# METRIC COLLECTOR - records everything during agent execution
# ===================================================================

class AgentMetrics:
    """
    Collects metrics at all three evaluation levels during agent runs.
    In production: ship these to Amazon CloudWatch.
    """
    def __init__(self):
        self.reset()

    def reset(self):
        self.runs = []
        self.current = None

    def start_run(self, task_id: str, task: str):
        self.current = {
            "task_id":             task_id,
            "task":                task,
            "start_time":          time.time(),
            # System-level
            "total_duration_s":    0,
            "llm_calls":           0,
            "llm_errors":          0,
            "input_tokens":        0,
            "output_tokens":       0,
            # Task-level
            "completed":           False,
            "success":             False,
            "human_intervention":  False,
            "turn_count":          0,
            # Tool-level
            "tool_calls":          [],
        }

    def record_llm_call(self, input_tokens: int, output_tokens: int, error: bool = False):
        if not self.current: return
        self.current["llm_calls"] += 1
        self.current["input_tokens"]  += input_tokens
        self.current["output_tokens"] += output_tokens
        if error:
            self.current["llm_errors"] += 1

    def record_tool_call(self, tool_name: str, inputs: dict,
                         correct_tool: str, correct_inputs: dict,
                         succeeded: bool):
        if not self.current: return
        # Tool selection accuracy: did it pick the right tool?
        correct_selection = (tool_name == correct_tool)
        # Parameter accuracy: did it pass the right inputs?
        correct_params = all(inputs.get(k) == v for k, v in correct_inputs.items())

        self.current["tool_calls"].append({
            "tool_called":         tool_name,
            "expected_tool":       correct_tool,
            "correct_selection":   correct_selection,
            "inputs_passed":       inputs,
            "expected_inputs":     correct_inputs,
            "correct_params":      correct_params,
            "succeeded":           succeeded,
        })

    def end_run(self, success: bool, completed: bool, human_needed: bool = False):
        if not self.current: return
        self.current["total_duration_s"] = round(time.time() - self.current["start_time"], 2)
        self.current["success"]            = success
        self.current["completed"]          = completed
        self.current["human_intervention"] = human_needed
        self.runs.append(self.current)
        self.current = None

    def summary(self):
        if not self.runs:
            print("No runs recorded yet.")
            return

        n = len(self.runs)
        print(f"\n{'='*55}")
        print(f"AGENT EVALUATION REPORT  ({n} tasks evaluated)")
        print(f"{'='*55}")

        # System-level metrics
        avg_duration  = sum(r["total_duration_s"] for r in self.runs) / n
        avg_llm_calls = sum(r["llm_calls"] for r in self.runs) / n
        total_tokens  = sum(r["input_tokens"] + r["output_tokens"] for r in self.runs)
        error_rate    = sum(r["llm_errors"] for r in self.runs) / max(1, sum(r["llm_calls"] for r in self.runs))
        cost_approx   = total_tokens * 0.000003  # approximate Claude Sonnet pricing per token

        print(f"\nSYSTEM-LEVEL METRICS (operational health)")
        print(f"  Avg task duration:   {avg_duration:.1f}s")
        print(f"  Avg LLM calls/task:  {avg_llm_calls:.1f}")
        print(f"  Total tokens used:   {total_tokens:,}")
        print(f"  Est. cost (approx):  ${cost_approx:.4f}")
        print(f"  LLM error rate:      {error_rate:.1%}")

        # Task-level metrics
        success_rate      = sum(1 for r in self.runs if r["success"]) / n
        completion_rate   = sum(1 for r in self.runs if r["completed"]) / n
        intervention_rate = sum(1 for r in self.runs if r["human_intervention"]) / n

        print(f"\nTASK-LEVEL METRICS (use-case performance)")
        print(f"  Agent success rate:      {success_rate:.0%}")
        print(f"  Task completion rate:    {completion_rate:.0%}")
        print(f"  Human intervention rate: {intervention_rate:.0%}  <- most important metric")

        # Interpret intervention rate
        if intervention_rate > 0.4:
            print(f"  WARNING: >40% intervention = supervised assistant, not autonomous agent")
        elif intervention_rate > 0.2:
            print(f"  CAUTION: >20% intervention - review failure patterns")
        else:
            print(f"  GOOD: <20% intervention - agent operating autonomously")

        # Tool-level metrics
        all_tool_calls = [t for r in self.runs for t in r["tool_calls"]]
        if all_tool_calls:
            selection_acc = sum(1 for t in all_tool_calls if t["correct_selection"]) / len(all_tool_calls)
            param_acc     = sum(1 for t in all_tool_calls if t["correct_params"]) / len(all_tool_calls)
            tool_success  = sum(1 for t in all_tool_calls if t["succeeded"]) / len(all_tool_calls)

            print(f"\nTOOL-LEVEL METRICS (API accuracy)")
            print(f"  Tool selection accuracy: {selection_acc:.0%}")
            print(f"  Parameter accuracy:      {param_acc:.0%}")
            print(f"  Tool success rate:       {tool_success:.0%}")
            print(f"  Total tool calls:        {len(all_tool_calls)}")

            # Diagnose problems
            if selection_acc < 0.8:
                print(f"  DIAGNOSIS: Poor selection accuracy -> review tool descriptions (ambiguous?)")
            if param_acc < 0.8:
                print(f"  DIAGNOSIS: Poor parameter accuracy -> review tool input schemas")
            if tool_success < 0.9:
                print(f"  DIAGNOSIS: Low success rate -> infrastructure issue, not agent issue")

        print(f"{'='*55}")

metrics = AgentMetrics()
print("Metrics collector ready")

## Step 3 - The Instrumented Agent

Same loop as all previous labs - with metric collection wired into every step.

In [ ]:
# ===================================================================
# INSTRUMENTED AGENT
# Same agent loop as all previous labs - but every step is measured
# ===================================================================

TOOLS = [
    {
        "name": "get_weather",
        "description": "Get current weather for a city. Returns temperature and conditions.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
            },
            "required": ["city"]
        }
    },
    {
        "name": "convert_currency",
        "description": "Convert an amount from one currency to another.",
        "input_schema": {
            "type": "object",
            "properties": {
                "amount": {"type": "number"},
                "from_currency": {"type": "string"},
                "to_currency": {"type": "string"}
            },
            "required": ["amount", "from_currency", "to_currency"]
        }
    },
    {
        "name": "calculate",
        "description": "Evaluate a mathematical expression.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"}
            },
            "required": ["expression"]
        }
    }
]

WEATHER_DATA = {
    "london": "14 degrees celsius, cloudy",
    "paris": "17 degrees celsius, sunny",
    "tokyo": "28 degrees celsius, humid",
    "new york": "22 degrees celsius, partly cloudy",
}
RATES = {"GBP": 1.0, "USD": 1.27, "EUR": 1.17, "JPY": 190.5}

def tool_executor(name: str, inputs: dict) -> tuple:
    """Returns (result_string, succeeded_bool)"""
    try:
        if name == "get_weather":
            city = inputs.get("city", "").lower()
            weather = WEATHER_DATA.get(city, f"Weather data not available for {inputs.get('city')}")
            return weather, True
        elif name == "convert_currency":
            frm = inputs.get("from_currency", "").upper()
            to  = inputs.get("to_currency", "").upper()
            if frm not in RATES or to not in RATES:
                return f"Currency not found: {frm} or {to}", False
            converted = (inputs["amount"] / RATES[frm]) * RATES[to]
            return f"{inputs['amount']} {frm} = {converted:.2f} {to}", True
        elif name == "calculate":
            safe = set("0123456789+-*/()., ")
            expr = inputs.get("expression", "")
            if not all(c in safe for c in expr):
                return "Error: invalid expression", False
            return f"Result: {eval(expr)}", True
        return f"Unknown tool: {name}", False
    except Exception as e:
        return f"Tool error: {e}", False


def run_instrumented_agent(
    task_id: str,
    task: str,
    expected_tool: str,
    expected_inputs: dict,
    should_succeed: bool = True
) -> str:
    """
    Agent that records metrics at every step.
    expected_tool and expected_inputs define the golden standard for evaluation.
    """
    metrics.start_run(task_id, task)
    messages = [{"role": "user", "content": task}]
    print(f"\n[{task_id}] {task[:60]}...")

    for _ in range(8):
        call_start = time.time()
        try:
            response = client.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=512,
                system="You are a helpful assistant. Use tools to answer questions accurately.",
                tools=TOOLS,
                messages=messages
            )
            # Record system-level: token usage
            usage = response.usage
            metrics.record_llm_call(
                input_tokens=usage.input_tokens,
                output_tokens=usage.output_tokens,
                error=False
            )
        except Exception as e:
            metrics.record_llm_call(0, 0, error=True)
            metrics.end_run(success=False, completed=False, human_needed=True)
            return f"LLM error: {e}"

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            final = next((b.text for b in response.content if hasattr(b, "text")), "")
            print(f"  Answer: {final[:80]}")
            metrics.end_run(success=True, completed=True, human_needed=False)
            return final

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result, succeeded = tool_executor(block.name, block.input)
                # Record tool-level metrics
                metrics.record_tool_call(
                    tool_name=block.name,
                    inputs=block.input,
                    correct_tool=expected_tool,
                    correct_inputs=expected_inputs,
                    succeeded=succeeded
                )
                print(f"  Tool: {block.name}({block.input}) -> {result[:50]}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })
        messages.append({"role": "user", "content": tool_results})

    metrics.end_run(success=False, completed=False, human_needed=True)
    return "Max turns"

print("Instrumented agent ready")

## Step 4 - Run the Evaluation Suite

8 test tasks. Each has a golden-standard expected tool and inputs.

In [ ]:
# ===================================================================
# EVALUATION SUITE - 8 test tasks with golden standards
# This is a mini eval harness. Production: AWS Step Functions + S3 for test sets
# ===================================================================

test_cases = [
    # (task_id, task, expected_tool, expected_key_inputs, should_succeed)
    ("T01", "What is the weather in London?",
     "get_weather", {"city": "London"}, True),

    ("T02", "How much is 250 GBP in US Dollars?",
     "convert_currency", {"amount": 250, "from_currency": "GBP", "to_currency": "USD"}, True),

    ("T03", "What is 144 divided by 12?",
     "calculate", {"expression": "144 / 12"}, True),

    ("T04", "I am flying to Tokyo from Paris. What is the weather in Tokyo?",
     "get_weather", {"city": "Tokyo"}, True),

    ("T05", "Convert 1000 EUR to Japanese Yen.",
     "convert_currency", {"amount": 1000, "from_currency": "EUR", "to_currency": "JPY"}, True),

    ("T06", "What is 15 percent of 840?",
     "calculate", {"expression": "840 * 0.15"}, True),

    ("T07", "Is it warmer in Paris or New York today?",
     "get_weather", {"city": "Paris"}, True),  # expects at least one weather call

    ("T08", "I have 500 USD. If I convert to GBP and then spend half, how much GBP do I have?",
     "convert_currency", {"amount": 500, "from_currency": "USD", "to_currency": "GBP"}, True),
]

print(f"Running evaluation suite: {len(test_cases)} tasks")
print("-" * 50)
metrics.reset()

for task_id, task, exp_tool, exp_inputs, exp_success in test_cases:
    run_instrumented_agent(task_id, task, exp_tool, exp_inputs, exp_success)

print("\nEvaluation complete. Generating report...")

In [ ]:
metrics.summary()

## Step 5 - Push to Amazon CloudWatch (Production Pattern)

In [ ]:
# ===================================================================
# HOW TO SHIP THESE METRICS TO AMAZON CLOUDWATCH
# ===================================================================
import boto3

print("""
In production: ship every metric to Amazon CloudWatch.
CloudWatch gives you dashboards, alerts, and anomaly detection.
""")

# Example CloudWatch metric push (requires CloudWatch permissions in IAM role)
def push_metrics_to_cloudwatch(metrics_obj, namespace="AgentBootcamp"):
    """
    Pushes evaluation metrics to CloudWatch.
    Run this after your evaluation suite completes.
    """
    cw = boto3.client("cloudwatch", region_name="us-east-1")
    run_data = metrics_obj.runs

    if not run_data:
        print("No metrics to push")
        return

    n = len(run_data)
    success_rate      = sum(1 for r in run_data if r["success"]) / n
    intervention_rate = sum(1 for r in run_data if r["human_intervention"]) / n
    avg_duration      = sum(r["total_duration_s"] for r in run_data) / n

    all_tools = [t for r in run_data for t in r["tool_calls"]]
    selection_acc = sum(1 for t in all_tools if t["correct_selection"]) / max(1, len(all_tools))

    metric_data = [
        {"MetricName": "AgentSuccessRate",       "Value": success_rate,      "Unit": "None"},
        {"MetricName": "HumanInterventionRate",  "Value": intervention_rate, "Unit": "None"},
        {"MetricName": "AvgTaskDurationSeconds", "Value": avg_duration,      "Unit": "Seconds"},
        {"MetricName": "ToolSelectionAccuracy",  "Value": selection_acc,     "Unit": "None"},
    ]

    # Uncomment to actually push:
    # cw.put_metric_data(Namespace=namespace, MetricData=metric_data)
    # print(f"Pushed {len(metric_data)} metrics to CloudWatch namespace: {namespace}")

    print("CloudWatch push (simulated - uncomment to enable):")
    for m in metric_data:
        print(f"  {m['MetricName']:35} = {m['Value']:.2%}")

push_metrics_to_cloudwatch(metrics)
print()
print("CloudWatch Alarms to set up in production:")
print("  HumanInterventionRate > 0.25  -> SNS alert to engineering team")
print("  AgentSuccessRate < 0.80       -> PagerDuty incident")
print("  AvgTaskDuration > 30s         -> Review LLM call count per task")
print("  ToolSelectionAccuracy < 0.85  -> Review tool descriptions")

## Key Takeaways

| Metric | Level | What a bad score tells you |
|--------|-------|--------------------------|
| Avg task duration | System | Too many LLM calls per task - simplify the task or tool descriptions |
| LLM error rate | System | Edge cases in prompts hitting model limits |
| Human intervention rate | Task | Core autonomy measure - >40% means it is not truly automated |
| Agent success rate | Task | How often the agent reaches a valid end state |
| Tool selection accuracy | Tool | Tool descriptions are ambiguous |
| Parameter accuracy | Tool | Tool input schemas are poorly specified |
| Tool success rate | Tool | Infrastructure problem, not agent problem |

**In production:**
- Run this eval suite on every code change before deployment
- Set CloudWatch alarms on human intervention rate and tool selection accuracy
- Track trends over time - a drift in accuracy usually means a tool description became ambiguous
  as the codebase evolved

---
**Next:** Lab 15 - API Internals: Tokenisation to Response Object
The final lab. You will see every stage of what happens inside the API call.